# SS3 vs IEDB Residue-Level Correlation

Tests whether predicted SS3 secondary-structure labels correlate with IEDB epitope masks at the residue level, using the curated positive sets from notebook 01. For each of 374 proteins, per-protein AUROC is computed for `structured_score` (H/E = α-helix/β-strand) vs `non-structured_score` (C = loops, turns, coil), then compared with a paired Wilcoxon signed-rank test.


In [ ]:
import gzip
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score

def _find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'pyproject.toml').exists() or (p / 'src' / 'xallergen').exists():
            return p
    raise FileNotFoundError('Repo root not found')

REPO_ROOT    = _find_repo_root(Path.cwd())
SRC_DIR      = REPO_ROOT / 'src'
DATA_DIR     = REPO_ROOT / 'data'
SS3_DIR      = DATA_DIR / 'ss3'
ANALYSIS_DIR = REPO_ROOT / 'results' / 'ss3_iedb_correlation'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from xallergen.baseline_notebook_utils import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
sns.set_theme(style='whitegrid', context='talk')

POSITIVES_SPLITA_CSV       = DATA_DIR / 'positives_splitA.csv'
POSITIVES_SPLITB_CSV       = DATA_DIR / 'positives_splitB.csv'
IEDB_FASTA_PATH            = SS3_DIR  / 'iedb_positive_sequences.fasta'
STRUCTURED_JSON_CANDIDATES = [SS3_DIR / 'iedb_positive_ss3_structured.json.gz',
                               SS3_DIR / 'iedb_positive_ss3_structured.json']
Q3_CSV_CANDIDATES          = [SS3_DIR / 'iedb_positive_ss3_q3.csv.gz',
                               SS3_DIR / 'iedb_positive_ss3_q3.csv',
                               SS3_DIR / 'iedb_positive_sequences_predictions.csv']
PER_PROTEIN_CSV = ANALYSIS_DIR / 'ss3_iedb_correlation_per_protein.csv'
FIGURE_PNG      = ANALYSIS_DIR / 'ss3_iedb_correlation_metrics.png'


In [ ]:
def _parse_intervals(value):
    if pd.isna(value): return []
    return [int(p.strip()) for p in str(value).split(';') if str(p).strip()]

def build_epitope_mask(sequence, starts, ends):
    seq  = str(sequence).strip().upper()
    mask = np.zeros(len(seq), dtype=np.float32)
    for s, e in zip(_parse_intervals(starts), _parse_intervals(ends)):
        mask[s - 1:e] = 1.0
    return mask

def write_fasta(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as fh:
        for row in frame[['accession', 'sequence']].drop_duplicates().itertuples(index=False):
            fh.write(f'>{row.accession}\n{row.sequence}\n')

def _find_column(cols, candidates):
    low = {c.strip().lower(): c.strip() for c in cols}
    for c in candidates:
        if c.lower() in low: return low[c.lower()]
    raise KeyError(f'None of {candidates} found in columns')

def _load_structured_from_q3(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    id_col = _find_column(df.columns, ('id', 'sequence_id', 'name', 'identifier', 'accession'))
    q3_col = _find_column(df.columns, ('q3', 'ss3', 'predicted_q3', 'secondary_structure'))
    df[id_col] = df[id_col].astype(str).str.strip().str.removeprefix('>')
    return {
        acc: [1.0 if str(v).strip().upper() in {'H', 'E'} else 0.0 for v in grp[q3_col]]
        for acc, grp in df.groupby(id_col, sort=False)
    }

def load_ss3():
    for p in STRUCTURED_JSON_CANDIDATES:
        if p.exists():
            opener = gzip.open(p, 'rt') if p.suffix == '.gz' else p.open()
            with opener as fh:
                raw = json.load(fh)
            return {str(k).strip(): [float(v) for v in vs] for k, vs in raw.items()}
    for p in Q3_CSV_CANDIDATES:
        if p.exists():
            return _load_structured_from_q3(p)
    raise FileNotFoundError('No SS3 predictions found — run NetSurfP on ' + str(IEDB_FASTA_PATH))


In [ ]:
split_a = pd.read_csv(POSITIVES_SPLITA_CSV).assign(split_name='splitA')
split_b = pd.read_csv(POSITIVES_SPLITB_CSV).assign(split_name='splitB')
iedb_df = pd.concat([split_a, split_b], ignore_index=True)
iedb_df['accession'] = iedb_df['accession'].astype(str).str.strip()
iedb_df['sequence']  = iedb_df['sequence'].astype(str).str.strip().str.upper()
iedb_df['epitope_mask'] = [
    build_epitope_mask(seq, s, e)
    for seq, s, e in zip(iedb_df['sequence'], iedb_df['epitope_start'], iedb_df['epitope_end'])
]
write_fasta(iedb_df, IEDB_FASTA_PATH)

ss3_lookup = load_ss3()
iedb_df['ss3_structured'] = iedb_df['accession'].map(ss3_lookup)

print(f'Loaded {len(iedb_df)} proteins')


In [ ]:
rows = []
for row in iedb_df.itertuples(index=False):
    mask = np.asarray(row.epitope_mask)
    ss3  = np.asarray(row.ss3_structured)
    if 0 < mask.sum() < len(mask):
        rows.append({'accession': row.accession,
                     'auroc': float(roc_auc_score(mask, ss3))})

auroc_df = pd.DataFrame(rows)
auroc_df.to_csv(PER_PROTEIN_CSV, index=False)
print(f'{len(auroc_df)} proteins | '
      f'mean structured AUROC = {auroc_df["auroc"].mean():.3f} | '
      f'mean non-structured AUROC = {1 - auroc_df["auroc"].mean():.3f}')


In [ ]:
from matplotlib.lines import Line2D
from scipy.stats import wilcoxon as _wilcoxon


def _pval_stars(p: float) -> str:
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def _sig_bracket(ax, x1, x2, y, h, text, fontsize=11):
    ax.plot([x1, x1, x2, x2], [y - h*0.5, y, y, y - h*0.5], lw=0.9, c='k', clip_on=False)
    ax.text((x1+x2)/2, y + h*0.15, text, ha='center', va='bottom',
            fontsize=fontsize, fontweight='bold', clip_on=False)


_, pval = _wilcoxon(auroc_df['auroc'], 1 - auroc_df['auroc'], alternative='greater')
stars   = _pval_stars(pval)

plot_df = pd.concat([
    auroc_df[['accession', 'auroc']].assign(score_label='Structured (H/E)'),
    auroc_df[['accession']].assign(auroc=1 - auroc_df['auroc'].values,
                                   score_label='Non-structured (C)'),
], ignore_index=True)

ORDER   = ['Structured (H/E)', 'Non-structured (C)']
PALETTE = {'Structured (H/E)': '#0f766e', 'Non-structured (C)': '#94a3b8'}

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
})

fig, ax = plt.subplots(figsize=(4.5, 5))
fig.subplots_adjust(bottom=0.15)

sns.boxplot(data=plot_df, x='score_label', y='auroc', hue='score_label',
            order=ORDER, hue_order=ORDER, palette=PALETTE,
            width=0.5, fliersize=1.2, linewidth=0.8, legend=False, ax=ax)

ax.axhline(0.5, color='#475569', linestyle='--', linewidth=0.9, alpha=0.8)

all_vals = plot_df['auroc'].values
y_min    = all_vals.min()
y_range  = all_vals.max() - y_min
y_top    = all_vals.max() + y_range * 0.06
_sig_bracket(ax, 0, 1, y_top, y_range * 0.03, stars)

ax.set_xlabel('')
ax.set_ylabel('Per-protein AUROC')
ax.set_ylim(y_min - y_range * 0.04, y_top + y_range * 0.16)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(
    handles=[Line2D([0], [0], color='#475569', linestyle='--', linewidth=0.9, label='Random (0.5)')],
    frameon=False, fontsize=9, loc='lower center',
)

fig.suptitle(
    'SS3 vs IEDB Epitopes: Per-Protein Residue Ranking\n'
    f'(n = {len(auroc_df)} proteins, paired Wilcoxon p = {pval:.2e})',
    fontsize=11, y=1.02,
)
fig.savefig(FIGURE_PNG, dpi=220, bbox_inches='tight')
plt.show()
print(f'p = {pval:.2e}  ({stars})')
print(f'Saved figure to: {FIGURE_PNG}')


## Interpretation

### What was tested
For each of 374 IEDB-positive proteins (splits A + B), SS3-predicted secondary-structure labels were used to rank residues and compared against the experimental IEDB epitope mask. Q3 prediction collapses into two scores: **structured** (H = α-helix, E = β-strand) and **non-structured** (C = loops, turns, and random coil). Per-protein AUROC was computed for each score, then compared across the two groups with a one-sided paired Wilcoxon signed-rank test (structured > non-structured).

Note: because non-structured score = 1 − structured score, AUROC(non-structured) = 1 − AUROC(structured) for every protein. The paired test is therefore mathematically equivalent to a one-sample test of structured AUROCs against the chance level of 0.5.

### Key result
Structured residues (H/E) rank IEDB epitope residues significantly above chance, while non-structured residues (C) rank them below chance (mean AUROC 0.553 vs 0.447, paired Wilcoxon p < 10⁻²⁰).

### Effect size and practical meaning
The effect is **statistically robust but practically small** (ΔAUROC ≈ 0.05 from chance). SS3 alone cannot distinguish epitope from non-epitope residues at useful precision.

**Why structured regions are slightly enriched:**
- IEDB epitopes derive from proteolytically processed peptides, which can in principle come from any secondary structure. However, α-helices and β-strands tend to be evolutionarily conserved and surface-accessible — traits correlated with immunodominance — which may weakly bias the curated epitope set toward structured residues.
- The non-structured class (C) is a heterogeneous catch-all covering loops, turns, and random coil. Antigenic loops are common in real epitopes, which likely attenuates the signal. A finer 8-class (DSSP) breakdown would be needed to disentangle antigenic loops from buried random coil.

**Implication for XAllergen2.0:**
SS3 captures a real but small independent signal. Its value in the model is as an auxiliary structural inductive bias — it should be included but kept down-weighted relative to sequence-derived features.
